# 2 — Construcción de Silver

Silver tipa, deduplica y valida. Los registros aceptados se integran con `MERGE`; los rechazados quedan en cuarentena con una causa explícita. La corrección de la transacción 42 permite observar un `UPDATE` real.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
%run ../common/quality_rules

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
bronze_all = qualified_table(config, "bronze_transactions_all")
assert spark.catalog.tableExists(bronze_all), "Falta bronze_transactions_all. Ejecutá la tarea 01."
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
quarantine = qualified_table(config, "silver_transactions_quarantine")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {customers} USING DELTA AS
SELECT customer_id, country, segment, created_date, email
FROM (
  SELECT try_cast(customer_id AS BIGINT) customer_id, country, segment,
         try_cast(created_date AS DATE) created_date, lower(trim(email)) email,
         row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_customers')}
) WHERE rn = 1 AND customer_id IS NOT NULL
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {products} USING DELTA AS
SELECT product_id, category, price
FROM (
  SELECT try_cast(product_id AS BIGINT) product_id, category, try_cast(price AS DECIMAL(12,2)) price,
         row_number() OVER (PARTITION BY product_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_products')}
) WHERE rn = 1 AND product_id IS NOT NULL AND price > 0
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
raw_columns = ["transaction_id", "customer_id", "product_id", "event_ts", "amount", "payment_channel", "device_id", "is_fraud", "source_batch_id", "updated_at"]
typed = add_transaction_types(spark.table(bronze_all).dropDuplicates(raw_columns))
known_customers = spark.table(customers).select(F.col("customer_id").alias("known_customer"))
known_products = spark.table(products).select(F.col("product_id").alias("known_product"))
checked = (typed
    .join(known_customers, F.col("customer_id_typed") == F.col("known_customer"), "left")
    .join(known_products, F.col("product_id_typed") == F.col("known_product"), "left"))
checked = add_quality_reason(checked)
record_key = F.coalesce(F.col("transaction_id_typed").cast("string"), F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("<NULL>")) for c in raw_columns]), 256))
latest = (checked.withColumn("_record_key", record_key)
    .withColumn("_rn", F.row_number().over(Window.partitionBy("_record_key").orderBy(F.col("updated_at_typed").desc_nulls_last(), F.col("source_batch_id").desc())))
    .where("_rn = 1"))

In [0]:
valid = (latest.where(F.col("quality_reason").isNull()).select(
    F.col("transaction_id_typed").alias("transaction_id"),
    F.col("customer_id_typed").alias("customer_id"),
    F.col("product_id_typed").alias("product_id"),
    F.col("event_ts_typed").alias("event_ts"),
    F.col("amount_typed").alias("amount"), "payment_channel", "device_id",
    F.col("is_fraud_typed").alias("is_fraud"), "source_batch_id",
    F.col("updated_at_typed").alias("updated_at"), F.current_timestamp().alias("processed_at")))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {transactions} (
 transaction_id BIGINT, customer_id BIGINT, product_id BIGINT, event_ts TIMESTAMP, amount DECIMAL(12,2),
 payment_channel STRING, device_id STRING, is_fraud INT, source_batch_id STRING, updated_at TIMESTAMP, processed_at TIMESTAMP
) USING DELTA""")
valid.createOrReplaceTempView("_valid_transactions")
spark.sql(f"""MERGE INTO {transactions} t USING _valid_transactions s ON t.transaction_id = s.transaction_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
rejected = latest.where(F.col("quality_reason").isNotNull()).select(
    *raw_columns, "quality_reason", F.current_timestamp().alias("rejected_at"))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {quarantine} (
 transaction_id STRING, customer_id STRING, product_id STRING, event_ts STRING, amount STRING,
 payment_channel STRING, device_id STRING, is_fraud STRING, source_batch_id STRING, updated_at STRING,
 quality_reason STRING, rejected_at TIMESTAMP
) USING DELTA""")
rejected.createOrReplaceTempView("_rejected_transactions")
spark.sql(f"""MERGE INTO {quarantine} t USING _rejected_transactions s
ON t.transaction_id <=> s.transaction_id AND t.source_batch_id <=> s.source_batch_id AND t.quality_reason = s.quality_reason
WHEN NOT MATCHED THEN INSERT *""")
display(spark.table(quarantine).groupBy("source_batch_id", "quality_reason").count().orderBy("source_batch_id", "quality_reason"))
display(spark.table(transactions).groupBy("source_batch_id").count().orderBy("source_batch_id"))
display(spark.sql(f"DESCRIBE HISTORY {transactions}"))

source_batch_id,quality_reason,count
batch_002,INVALID_AMOUNT,1
batch_002,UNKNOWN_CUSTOMER,1
batch_003,INVALID_AMOUNT,1
batch_003,UNKNOWN_CUSTOMER,1
initial,INVALID_AMOUNT,51


source_batch_id,count
batch_002,200
batch_003,201
initial,49948


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-10-03T14:08:25.000Z,77401094317247,maripena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#12149L = transaction_id#12127L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#12133 > updated_at#12158)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(621996950576258),7c5a6643-f35e-4aef-8ad4-b5a35b29131c,1003-131232-ovnn9sk9-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 9024, materializeSourceTimeMs -> 2827, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3808, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50349, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2253)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-03T13:26:29.000Z,77401094317247,maripena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#12677L = transaction_id#12655L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#12661 > updated_at#12686)"",""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(525285604866426, bigdata_belu__pena_silver_gold, 456827467225548, 21395717974356, 77401094317247, manual, 7474644315277246)",List(621996950576258),a64989a5-a314-4367-a3a1-1136b31bc54b,1003-132248-gci83eh3-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4741, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 11293, materializeSourceTimeMs -> 2704, numTargetRowsInserted -> 200, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4660, numTargetRowsUpdated -> 1, numOutputRows -> 201, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50349, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3775)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-03T13:02:07.000Z,77401094317247,maripena@itba.edu.ar,MERGE,"Map(predicate -> [""(transaction_id#24147L = transaction_id#24125L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#24131 > updated_at#24156)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(525285604866426, bigdata_belu__pena_silver_gold, 944154348338115, 370524621448522, 77401094317247, manual, 7474644315277246)",List(621996950576258),51623649-cee0-4f54-9a33-7e1139cde85a,1003-125043-l3wiq676-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4492, materializeSourceTimeMs -> 1512, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1549, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50149, numTargetFilesRemoved -> 0, numTarget